## Imports

In [1]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## Connection

In [2]:
presto_port = '80'
# username = 'airflow-user'
username = 'manoj.ravirajan@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3 = presto.connect('processing-2.processing.data.production.internal', presto_port, username)
connection4 = presto.connect('processing.processing.data.production.internal', presto_port, username)

## Parameter

In [3]:
city = 'Bangalore'
date_list = [ '2024-04-07', '2024-04-14'] 
# date_list = [ '2024-04-07', '2024-04-14', '2024-04-21', '2024-04-28', '2024-05-05']
base_week = date_list[0]

## Dataset & User-Defined Functions

In [4]:
def consideration_query(run_date, city_name, base_week):
    
    rr_data_query = f"""
  
        with dormant_customer as (
            
            select
                customer_id as customer,
                date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) rapido_first_ride,
                ceil(cast(date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) as real)/10)*10 rapido_age_seg_fr,
                
                taxi_lifetime_last_ride_date, taxi_lifetime_first_ride_date, 
                taxi_lifetime_stage, previous_taxi_lifetime_stage, taxi_income_segment,
                taxi_lifetime_rides, taxi_recency_segment, taxi_rr_active_days_last_21_days as run_taxi_rr_active_days_last_21_days,
                taxi_lifetime_last_ride_city city_name, 
                
                fe_mean_intent_type, fe_intent_trend_type, fe_potential_per_type, 
                fe_regularity_type, fe_recency_type, fe_volatility_type,
                
                gender_tag, ps_tag_link, ps_tag_auto,
                
                taxi_service_affinity, taxi_distance_preference, taxi_time_of_day_preference, taxi_day_of_week_preference
            
            from 
                datasets.iallocator_customer_segments
            where 
                date(run_date) = date('{base_week}')
                -- and taxi_lifetime_last_ride_date = date('2023-11-01')
                and taxi_recency_segment in ('RECENT')
                and taxi_lifetime_rides > 4
                and taxi_lifetime_last_ride_city in ('{city_name}')
        ),
        
        base as (
        
            select 
                a.*,
                case when coalesce(b.rr,0) <1 then 0 else 1 end as rr_retention_flag,
                case when coalesce(b.fe,0) <1 then 0 else 1 end as fe_retention_flag,
                case when coalesce(b.ao,0) <1 then 0 else 1 end as ao_retention_flag,
                case when coalesce(b.nr,0) <1 then 0 else 1 end as net_retention_flag,
                
                case when coalesce(b.nr_post_w01,0) <1 then 0 else 1 end as nr_flag_w01,
                case when coalesce(b.nr_post_w02,0) <1 then 0 else 1 end as nr_flag_w02,
                case when coalesce(b.nr_post_w03,0) <1 then 0 else 1 end as nr_flag_w03,
                case when coalesce(b.nr_post_w04,0) <1 then 0 else 1 end as nr_flag_w04,
                case when coalesce(b.nr_post_w05,0) <1 then 0 else 1 end as nr_flag_w05,
                case when coalesce(b.nr_post_w06,0) <1 then 0 else 1 end as nr_flag_w06,
                case when coalesce(b.nr_post_w07,0) <1 then 0 else 1 end as nr_flag_w07,
                case when coalesce(b.nr_post_w08,0) <1 then 0 else 1 end as nr_flag_w08,
                case when coalesce(b.nr_post_w09,0) <1 then 0 else 1 end as nr_flag_w09,
                case when coalesce(b.nr_post_w10,0) <1 then 0 else 1 end as nr_flag_w10,
                
                case when coalesce(b.rr_post_w01,0) <1 then 0 else 1 end as rr_flag_w01,
                case when coalesce(b.rr_post_w02,0) <1 then 0 else 1 end as rr_flag_w02,
                case when coalesce(b.rr_post_w03,0) <1 then 0 else 1 end as rr_flag_w03,
                case when coalesce(b.rr_post_w04,0) <1 then 0 else 1 end as rr_flag_w04,
                case when coalesce(b.rr_post_w05,0) <1 then 0 else 1 end as rr_flag_w05,
                case when coalesce(b.rr_post_w06,0) <1 then 0 else 1 end as rr_flag_w06,
                case when coalesce(b.rr_post_w07,0) <1 then 0 else 1 end as rr_flag_w07,
                case when coalesce(b.rr_post_w08,0) <1 then 0 else 1 end as rr_flag_w08,
                case when coalesce(b.rr_post_w09,0) <1 then 0 else 1 end as rr_flag_w09,
                case when coalesce(b.rr_post_w10,0) <1 then 0 else 1 end as rr_flag_w10,
                
                case when coalesce(b.fe_post_w01,0) <1 then 0 else 1 end as fe_flag_w01,
                case when coalesce(b.fe_post_w02,0) <1 then 0 else 1 end as fe_flag_w02,
                case when coalesce(b.fe_post_w03,0) <1 then 0 else 1 end as fe_flag_w03,
                case when coalesce(b.fe_post_w04,0) <1 then 0 else 1 end as fe_flag_w04,
                case when coalesce(b.fe_post_w05,0) <1 then 0 else 1 end as fe_flag_w05,
                case when coalesce(b.fe_post_w06,0) <1 then 0 else 1 end as fe_flag_w06,
                case when coalesce(b.fe_post_w07,0) <1 then 0 else 1 end as fe_flag_w07,
                case when coalesce(b.fe_post_w08,0) <1 then 0 else 1 end as fe_flag_w08,
                case when coalesce(b.fe_post_w09,0) <1 then 0 else 1 end as fe_flag_w09,
                case when coalesce(b.fe_post_w10,0) <1 then 0 else 1 end as fe_flag_w10

            from 
                dormant_customer a
            left join
                (
                select 
                    customerid,
                    -- sum(ao_sessions_unique_daily) ao,
                    -- sum(fe_sessions_unique_daily) fe,
                    -- sum(rr_sessions_unique_daily) rr,
                    -- sum(net_rides_daily) nr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then ao_sessions_unique_daily end) as ao,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w01,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w01,
                    
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w01

                from 
                    datasets.customer_rf_daily_kpi 
                where 
                    date(day) > date('{base_week}') 
                    and date(day) <= date('{base_week}') + interval '70' day
                    and service_name in ( 'Auto', 'Auto NCR', 'Auto Pool' , 'Link', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'CabEconomy', 'CabPremium')
                    and city in ('{city_name}')
                group by 1
                ) b
                on a.customer=b.customerid

        ),
    
        rr_data as (
        
            select 
                fe.*
            from
                (
                select 
                    aa.*, 
                    bb.rr_LT, 
                    bb.rapido_age,
                    bb.rapido_age_seg
                from
                    (
                    select 
                        customerid,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w13,
                        sum(case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w12,
                        sum(case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w11,
                        sum(case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w10,
                        sum(case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w09,
                        sum(case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w08,
                        sum(case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w07,
                        sum(case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w06,
                        sum(case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w05,
                        sum(case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w04,
                        sum(case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w03,
                        sum(case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w02,
                        sum(case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w01,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_91days,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_91days,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then day end) as active_rr_days_w13,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then day end) as active_rr_days_w12,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then day end) as active_rr_days_w11,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then day end) as active_rr_days_w10,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then day end) as active_rr_days_w09,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then day end) as active_rr_days_w08,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then day end) as active_rr_days_w07,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then day end) as active_rr_days_w06,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then day end) as active_rr_days_w05,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then day end) as active_rr_days_w04,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then day end) as active_rr_days_w03,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then day end) as active_rr_days_w02,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then day end) as active_rr_days_w01,
                        day((date('{run_date}') + interval '-01' day) - date(max(day))) as days_since_last_rr
                    from 
                        (
                        select 
                            customerid,day, 
                            max(fe_sessions_unique_daily) as fe_sessions_unique_daily,  
                            sum(rr_sessions_unique_daily) as rr_sessions_unique_daily
                        from 
                            datasets.customer_rf_daily_kpi a
                        join
                            base b
                            on a.customerid = b.customer
                            and 1 = 1 
                            and city = '{city_name}'
                            and rr_sessions_unique_daily >= 1 
                            and day >= date_format(date('{run_date}') + interval '-91' day , '%Y-%m-%d')
                            and day <= date_format(date('{run_date}') + interval '-1' day , '%Y-%m-%d')
                            and service_name in ( 'Auto', 'Auto NCR', 'Auto Pool' , 'Link', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'CabEconomy', 'CabPremium')
                        group by customerid, day
                        )
                    group by customerid
                    ) aa 
                left join 
                    (
                    select 
                        customerid, 
                        sum(rr_sessions_unique_daily) as rr_LT , 
                        day((date('{run_date}') + interval '-01' day) - date(min(day))) as rapido_age,
                        ceil(cast(day((date('{run_date}') + interval '-01' day) - date(min(day))) as real)/10)*10 rapido_age_seg
                    from 
                        datasets.customer_rf_daily_kpi a
                    join
                        base b
                        on a.customerid=b.customer
                        and rr_sessions_unique_daily >= 1 
                        and city = '{city_name}'
                    group by 1
                    ) bb
                    on aa.customerid = bb.customerid
                ) fe 
        )
        
        select 
            a.*, b.*,
            '{run_date}' as run_date
        from 
            base a
        left join
            rr_data b
            on a.customer=b.customerid
    """

    try:
        data = pd.read_sql(rr_data_query, connection3)
    except:
        data = pd.read_sql(rr_data_query, connection4)
        
    
    ## Data fetched successfully
    print(data.shape)


    ## .mask(data[columns_to_replace] == 0, pd.NA) replaces all 0 values in these columns with pd.NA.
    columns_to_replace = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11','active_rr_days_w10', 'active_rr_days_w09',
                          'active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06', 'active_rr_days_w05', 'active_rr_days_w04',
                          'active_rr_days_w03', 'active_rr_days_w02', 'active_rr_days_w01']
    data[columns_to_replace] = data[columns_to_replace].mask(data[columns_to_replace] == 0, pd.NA)
    

    ## Unused
    ## RR-Normalization = Total no. of RR/Active days
    """
    data['rr_norm_w13'] = data.rr_w13/data.active_rr_days_w13
    data['rr_norm_w12'] = data.rr_w12/data.active_rr_days_w12
    data['rr_norm_w11'] = data.rr_w11/data.active_rr_days_w11
    data['rr_norm_w10'] = data.rr_w10/data.active_rr_days_w10
    data['rr_norm_w09'] = data.rr_w09/data.active_rr_days_w09
    data['rr_norm_w08'] = data.rr_w08/data.active_rr_days_w08
    data['rr_norm_w07'] = data.rr_w07/data.active_rr_days_w07
    data['rr_norm_w06'] = data.rr_w06/data.active_rr_days_w06
    data['rr_norm_w05'] = data.rr_w05/data.active_rr_days_w05
    data['rr_norm_w04'] = data.rr_w04/data.active_rr_days_w04
    data['rr_norm_w03'] = data.rr_w03/data.active_rr_days_w03
    data['rr_norm_w02'] = data.rr_w02/data.active_rr_days_w02
    data['rr_norm_w01'] = data.rr_w01/data.active_rr_days_w01
    """
    
    raw_customer_data = data[data['customerid']!=0]
    back_up = raw_customer_data.copy() ## With NA as value
    fe_data2 = raw_customer_data.fillna(0)
    fe_data = fe_data2[fe_data2['customerid']!=0] ## With 0 as value
    
    
    ## Consistency/Regularity Threshold 
    ## RR Regularity → Count of active weeks in last 13 weeks/ (13 or Rapido age in weeks if a customer is less than 13 weeks old)
    ## Active Days Mean → Total active days in last 13 weeks/ count of active weeks in last 13 weeks
    
    def regularity_decider_v2(x,active_mean):
        
        if x<=0.2:
            return '05_rare_need'
        elif x>0.2 and x<=0.4:
            return '04_monthly'
        elif x>0.4 and x<=0.689:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'


    ## Intent Threshold
    ## Intent → Total RR in last 13 weeks/ No. of Active weeks
    def intent_decider(x):
        
        if x<=2:
            return '03_Low'
        elif x>2 and x<=5:
            return '02_Medium'
        else:
            return '01_High'
    

    ## Consideration signal
    def consideration_signal_creator(x,y,active_mean):
        
        if y<=1.5:
            if x<=0.3:
                return 'rarely_1_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_1_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_1_rr'
            else:
                return 'weekly_1_rr'
        else:
            if x<=0.3:
                return 'rarely_2+_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_2+_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_2+_rr'
            else:
                return 'weekly_2+_rr'
                

    ## Correction factor
    def correction_factor(rr_weight,rapido_age):
        
        if rapido_age >= 91:
            week_inactive = 0
            multiplier = 1

        elif rapido_age == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))

        # 7,14,21,28,...
        elif rapido_age%7 == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))    
            
        else:
            week_inactive = (91-rapido_age)//7
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2))) 
        
        return (multiplier, 
                week_inactive, 
                (rr_weight.sum()- (week_inactive*(week_inactive+1)/2)), 
                ((rr_weight*rr_weight).sum() - (week_inactive*(week_inactive+1)*(2*week_inactive+1)/6))
               )

    
    ## Recency trend
    def recency_type(days_since_last_fe):
        
        if days_since_last_fe <= 6: 
            return('Recent')
        elif days_since_last_fe <= 20:
            return('Stationary')
        else:
            return('Dormant')

    ## Intent trend
    def intent_trend_type(intent_trend):
        
        if intent_trend <= -0.1:
            return('03_Declining')
        elif intent_trend <= 0.1:
            return('02_Stable')
        else:
            return('01_Increasing')
    
    
    ## Creating Final signal
    def create_L3_signals(fe_data, back_up): 
        
        print('Creating L3 Signal')
        rr_weight = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13])

        ## sum (rr_sessions_unique_daily)
        fe_cols = ['rr_w13', 'rr_w12', 'rr_w11', 'rr_w10', 'rr_w09','rr_w08', 'rr_w07', 
                   'rr_w06','rr_w05', 'rr_w04', 'rr_w03', 'rr_w02','rr_w01']   
        ## count (rr_days)
        activedays_cols = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11', 'active_rr_days_w10', 
                           'active_rr_days_w09','active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06',
                           'active_rr_days_w05', 'active_rr_days_w04', 'active_rr_days_w03','active_rr_days_w02', 
                           'active_rr_days_w01'] 
        
        
        print('Calctn-1')
        fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight,x)[0])
        
        fe_data['rr_mean_intent'] =  back_up[fe_cols].mean(axis=1) ## Check logic once
        fe_data['rr_mean_intent_final'] = fe_data['rr_mean_intent'].apply(lambda x: intent_decider(x))

        fe = fe_data[fe_cols]
        fe = fe.ge(1.0).astype(int)
        
        # This multiplies the sum of the weighted values by 1/91 to normalize the result.
        # rr_w13 to rr_w01 || rr_weight 1 to 13
        fe_data['rr_regularity'] = (1/91)*(np.array(fe)*rr_weight).sum(axis=1)
        fe_data['rr_regularity_scaled'] = fe_data['rr_regularity']*fe_data['rr_intent_multiplier']
        
        fe_data['active_days_mean'] =  back_up[activedays_cols].mean(axis=1) ## Check logic once
        
        print('Calctn-2')
        fe_data['rr_regularity_final'] = fe_data.apply(lambda x: regularity_decider_v2(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        fe_data['reg_intent'] = fe_data.rr_regularity_final+'_'+fe_data.rr_mean_intent_final
        
        return(fe_data)

    
    print('UDF-Created Successfully')
    
    
    L3_signals = create_L3_signals(fe_data,back_up)
    L3_signals['city'] = city_name
    
    final_data = L3_signals.copy()
    
    return final_data    

## Signal Generation

In [5]:
concatenated_df=pd.DataFrame()

for i in date_list:
    
    print(i)
    tdf = consideration_query(i, city, base_week)
    
    print(tdf.shape[0])
    concatenated_df=pd.concat([concatenated_df, tdf]).reset_index(drop=True)

2024-04-07
(673753, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
649671
2024-04-14
(673753, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
652976


In [6]:
concatenated_df['reg_intent'] = concatenated_df.rr_regularity_final + '_' + concatenated_df.rr_mean_intent_final
base_df = concatenated_df[concatenated_df.run_date == base_week].reset_index(drop=True)

In [7]:
concatenated_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_intent_multiplier,rr_mean_intent,rr_mean_intent_final,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,5dac9c697431b2279cc02050,696,700.0,2024-04-04,2022-05-12,COMMITTED,HOOK,HIGH_INCOME,424,RECENT,11,Bangalore,High,Increasing,High,High,Recent,High,MALE,UNKNOWN,NPS,ONLY_AUTO,SHORT_DISTANCE,EVENING_PEAK,UNKNOWN,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,5dac9c697431b2279cc02050,1.0,9.0,6.0,4.0,3.0,8.0,2.0,14.0,9.0,8.0,0.0,12.0,11.0,87.0,83.0,1.0,3.0,5.0,3.0,1.0,5.0,1.0,6.0,5.0,2.0,0.0,6.0,5.0,2.0,865.0,695.0,700.0,2024-04-07,1.000000,7.25,01_High,0.879121,0.879121,3.583333,01_daily,01_daily_01_High,Bangalore
1,62c91d339adb4e5067305179,569,570.0,2024-04-07,2022-09-16,HOOK,DORMANT,UNKNOWN,14,RECENT,2,Bangalore,Low,Stable,High,Low,Recent,High,MALE,UNKNOWN,UNKNOWN,ONLY_LINK,MEDIUM_DISTANCE,MORNING_PEAK,WEEKEND,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,62c91d339adb4e5067305179,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,19.0,637.0,640.0,2024-04-07,1.000000,1.00,03_Low,0.241758,0.241758,1.000000,04_monthly,04_monthly_03_Low,Bangalore
2,65ade3d138890c5526f97d33,76,80.0,2024-04-06,2024-01-22,CHURN_OTB,DORMANT,UNKNOWN,10,RECENT,1,Bangalore,Low,Declining,High,High,Recent,High,FEMALE,UNKNOWN,NPS,ONLY_AUTO,SHORT_DISTANCE,MORNING_PEAK,UNKNOWN,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,1,1,1,65ade3d138890c5526f97d33,0.0,0.0,2.0,2.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0,10.0,12.0,0.0,0.0,1.0,1.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,14.0,75.0,80.0,2024-04-07,1.034091,2.00,03_Low,0.351648,0.363636,1.400000,04_monthly,04_monthly_03_Low,Bangalore


In [8]:
base_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_intent_multiplier,rr_mean_intent,rr_mean_intent_final,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,5dac9c697431b2279cc02050,696,700.0,2024-04-04,2022-05-12,COMMITTED,HOOK,HIGH_INCOME,424,RECENT,11,Bangalore,High,Increasing,High,High,Recent,High,MALE,UNKNOWN,NPS,ONLY_AUTO,SHORT_DISTANCE,EVENING_PEAK,UNKNOWN,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,5dac9c697431b2279cc02050,1.0,9.0,6.0,4.0,3.0,8.0,2.0,14.0,9.0,8.0,0.0,12.0,11.0,87.0,83.0,1.0,3.0,5.0,3.0,1.0,5.0,1.0,6.0,5.0,2.0,0.0,6.0,5.0,2.0,865.0,695.0,700.0,2024-04-07,1.000000,7.25,01_High,0.879121,0.879121,3.583333,01_daily,01_daily_01_High,Bangalore
1,62c91d339adb4e5067305179,569,570.0,2024-04-07,2022-09-16,HOOK,DORMANT,UNKNOWN,14,RECENT,2,Bangalore,Low,Stable,High,Low,Recent,High,MALE,UNKNOWN,UNKNOWN,ONLY_LINK,MEDIUM_DISTANCE,MORNING_PEAK,WEEKEND,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,62c91d339adb4e5067305179,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,19.0,637.0,640.0,2024-04-07,1.000000,1.00,03_Low,0.241758,0.241758,1.000000,04_monthly,04_monthly_03_Low,Bangalore
2,65ade3d138890c5526f97d33,76,80.0,2024-04-06,2024-01-22,CHURN_OTB,DORMANT,UNKNOWN,10,RECENT,1,Bangalore,Low,Declining,High,High,Recent,High,FEMALE,UNKNOWN,NPS,ONLY_AUTO,SHORT_DISTANCE,MORNING_PEAK,UNKNOWN,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,1,1,1,65ade3d138890c5526f97d33,0.0,0.0,2.0,2.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0,10.0,12.0,0.0,0.0,1.0,1.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,14.0,75.0,80.0,2024-04-07,1.034091,2.00,03_Low,0.351648,0.363636,1.400000,04_monthly,04_monthly_03_Low,Bangalore


In [9]:
concatenated_df.shape

(1302647, 102)

In [10]:
concatenated_df.run_date.unique()

array(['2024-04-07', '2024-04-14'], dtype=object)

In [11]:
base_df.shape

(649671, 102)

In [12]:
base_df.run_date.unique()

array(['2024-04-07'], dtype=object)

In [13]:
# concatenated_df.to_csv('concatenated_df.csv', index=False)
# base_df.to_csv('base_df.csv', index=False)

# concatenated_df = pd.read_csv('concatenated_df.csv')
# base_df = pd.read_csv('base_df.csv')

## Stability Function

In [14]:
def stability_func(df,base_week):
    
    df_base_week = df[df.run_date == base_week][['customer','run_date','rr_regularity_final','rr_mean_intent_final','reg_intent']]\
                                                    .reset_index(drop=True)
    ranking = {'05_rare_need' : 1,
               '04_monthly' : 2,
               '03_bi_weekly' : 3,
               '02_weekly' : 4,
               '01_daily' : 5
              }

    intent_ranking = {'03_Low' : 1,
                      '02_Medium' : 2,
                      '01_High' : 3
                     }

    final_rank = {'05_rare_need_03_Low' : 1,
                  '05_rare_need_02_Medium' : 2,
                  '05_rare_need_01_High' : 3,
                  '04_monthly_03_Low' : 4,
                  '04_monthly_02_Medium' : 5,
                  '04_monthly_01_High' : 6,
                  '03_bi_weekly_03_Low' : 7,
                  '03_bi_weekly_02_Medium' : 8,
                  '03_bi_weekly_01_High' : 9,
                  '02_weekly_03_Low' : 10,
                  '02_weekly_02_Medium' : 11,
                  '02_weekly_01_High' : 12,
                  '01_daily_02_Medium' : 13,
                  '01_daily_01_High' : 14
                 }    

    df['reg_rank'] = df['rr_regularity_final'].map(ranking)
    df['int_rank'] = df['rr_mean_intent_final'].map(intent_ranking)
    df['final_rank'] = df['reg_intent'].map(final_rank)

    
    
    ## Regularity Stabillity
    pvt_df = pd.pivot_table(df, values='reg_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df.columns
    dropped_reg = pvt_df.iloc[:, 1:].lt(pvt_df[base_week], axis=0)

    # Display the result
    print(dropped_reg.head())
    dropped_reg.head()
    first_dropped_week = dropped_reg.idxmax(axis=1)

    pvt_df['first_dropped_week'] = first_dropped_week.where(dropped_reg.any(axis=1), 'Maintained/Upgraded')

    pvt_df.reset_index(inplace=True)
    pvt_df.head()

    df_base_week_stability = pvt_df.merge(df_base_week, on=['customer'], how='inner')

    reg_stab = df_base_week_stability\
                        .groupby(by=['run_date','rr_regularity_final','first_dropped_week'], as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_regularity_final',], 
                                     columns='first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['rr_regularity_final'],as_index=False)\
                                   .agg(base=('customer','nunique')), on=['rr_regularity_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/regularity_stability.csv',index=False)

    
    
    
    ## Intent Stabillity
    pvt_df_int=pd.pivot_table(df, values='int_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_int.columns
    dropped_int = pvt_df_int.iloc[:, 1:].lt(pvt_df_int[base_week], axis=0)

    # Display the result
    print(dropped_int.head())
    dropped_int.head()
    int_first_dropped_week = dropped_int.idxmax(axis=1)

    pvt_df_int['int_first_dropped_week'] = int_first_dropped_week.where(dropped_int.any(axis=1), 'Maintained/Upgraded')

    pvt_df_int.reset_index(inplace=True)
    pvt_df_int.head()

    df_base_week_int_stability = pvt_df_int.merge(df_base_week, on=['customer'], how='inner')

    int_stab = df_base_week_int_stability\
                        .groupby(by=['run_date','rr_mean_intent_final','int_first_dropped_week'],as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_mean_intent_final',], 
                                     columns='int_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                    .groupby(by=['rr_mean_intent_final'], as_index=False)\
                                    .agg(base=('customer','nunique')), on=['rr_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/intent_stability.csv',index=False)

    
    
    
    ## Final Stabillity
    pvt_df_final=pd.pivot_table(df, values='final_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_final.columns
    dropped_final = pvt_df_final.iloc[:, 1:].lt(pvt_df_final[base_week], axis=0)

    
    # Display the result
    print(dropped_final.head())
    dropped_final.head()
    final_first_dropped_week = dropped_final.idxmax(axis=1)

    pvt_df_final['final_first_dropped_week'] = final_first_dropped_week.where(dropped_final.any(axis=1), 'Maintained/Upgraded')

    pvt_df_final.reset_index(inplace=True)
    pvt_df_final.head()

    df_base_week_final_stability = pvt_df_final.merge(df_base_week, on=['customer'], how='inner')

    final_stab = df_base_week_final_stability\
                        .groupby(by=['run_date','rr_regularity_final','rr_mean_intent_final','final_first_dropped_week'],
                                 as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_regularity_final','rr_mean_intent_final'], 
                                     columns='final_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['rr_regularity_final','rr_mean_intent_final'], as_index=False)\
                                   .agg(base=('customer','nunique')), on=['rr_regularity_final','rr_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/final_stability.csv',index=False)
    
    return reg_stab, int_stab, final_stab

In [15]:
reg_df, int_df, final_seg_df = stability_func(concatenated_df, base_week)

run_date  2024-04-07  2024-04-14
0              False       False
1              False       False
2              False       False
3              False       False
4              False       False
run_date  2024-04-07  2024-04-14
0              False       False
1              False       False
2              False       False
3              False       False
4              False       False
run_date  2024-04-07  2024-04-14
0              False       False
1              False       False
2              False       False
3              False       False
4              False       False


In [16]:
# complete flow
def stab_func_2(df, base_week, base_df, cons_type):
    
    # base_df=df[df.run_date==base_week].reset_index(drop=True)
    fdf1 = base_df.groupby(by=['city', 'run_date', cons_type], as_index=False).agg(base=('customer','nunique'))
    j = 0
    
    for i in date_list:
        
        j += 1
        if i != base_week:
            
            tdf = df[df.run_date==base_week][['run_date', 'customer', cons_type]]\
                                                .merge(df[df.run_date == i][['customer', cons_type]],on=['customer'],how='left')\
                                                .pivot_table(values='customer', 
                                                             index=['run_date',cons_type+'_x'], 
                                                             columns=cons_type+'_y', aggfunc='nunique')\
                                                .reset_index()\
                                                .fillna(0)
            
            # print(list(tdf))
            items_to_remove = ['run_date', cons_type+'_x']
            filtered_list = [x for x in list(tdf) if x not in items_to_remove]
            tdf.columns = ['run_date', cons_type]+[ j + '_' + i for j in filtered_list]
            fdf1 = fdf1.merge(tdf, on=['run_date', cons_type], how='inner')
    
    return fdf1
        
        

In [17]:
#cons_type: 'rr_regularity_final', 'rr_mean_intent_final', 'reg_intent'
finaldf = stab_func_2(concatenated_df, base_week, base_df, 'reg_intent')

In [18]:
finaldf

,city,run_date,reg_intent,base,01_daily_01_High_2024-04-14,01_daily_02_Medium_2024-04-14,02_weekly_01_High_2024-04-14,02_weekly_02_Medium_2024-04-14,02_weekly_03_Low_2024-04-14,03_bi_weekly_01_High_2024-04-14,03_bi_weekly_02_Medium_2024-04-14,03_bi_weekly_03_Low_2024-04-14,04_monthly_01_High_2024-04-14,04_monthly_02_Medium_2024-04-14,04_monthly_03_Low_2024-04-14,05_rare_need_01_High_2024-04-14,05_rare_need_02_Medium_2024-04-14,05_rare_need_03_Low_2024-04-14
0,Bangalore,2024-04-07,01_daily_01_High,77225,71484.0,2471.0,741.0,665.0,0.0,1809.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Bangalore,2024-04-07,01_daily_02_Medium,33364,3368.0,25156.0,20.0,3471.0,12.0,30.0,1307.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Bangalore,2024-04-07,02_weekly_01_High,4861,739.0,8.0,2687.0,757.0,0.0,637.0,33.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Bangalore,2024-04-07,02_weekly_02_Medium,116402,860.0,3860.0,1002.0,94424.0,2210.0,45.0,13714.0,287.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Bangalore,2024-04-07,02_weekly_03_Low,35259,39.0,41.0,34.0,4492.0,20472.0,0.0,223.0,9958.0,0.0,0.0,0.0,0.0,0.0,0.0
5,Bangalore,2024-04-07,03_bi_weekly_01_High,9519,2014.0,143.0,283.0,164.0,0.0,5727.0,817.0,1.0,347.0,23.0,0.0,0.0,0.0,0.0
6,Bangalore,2024-04-07,03_bi_weekly_02_Medium,92423,291.0,1502.0,132.0,13771.0,645.0,1072.0,65032.0,3627.0,35.0,6024.0,292.0,0.0,0.0,0.0
7,Bangalore,2024-04-07,03_bi_weekly_03_Low,91701,0.0,2.0,0.0,1584.0,6779.0,10.0,7008.0,63930.0,0.0,266.0,12122.0,0.0,0.0,0.0
8,Bangalore,2024-04-07,04_monthly_01_High,4256,0.0,0.0,0.0,0.0,0.0,1254.0,297.0,0.0,2243.0,333.0,2.0,111.0,15.0,1.0
9,Bangalore,2024-04-07,04_monthly_02_Medium,39019,0.0,0.0,0.0,0.0,0.0,450.0,10400.0,1310.0,448.0,22898.0,1760.0,16.0,1598.0,139.0


In [19]:
base_df\
.groupby(by=['city_name','run_date','reg_intent'],as_index=False)\
.agg(
    user_count=('customer','nunique'),
    fe_retention_w01=('fe_flag_w01','sum'),
    fe_retention_w02=('fe_flag_w02','sum'),
    fe_retention_w03=('fe_flag_w03','sum'),
    fe_retention_w04=('fe_flag_w04','sum'),
    fe_retention_w05=('fe_flag_w05','sum'),
    fe_retention_w06=('fe_flag_w06','sum'),
    fe_retention_w07=('fe_flag_w07','sum'),
    fe_retention_w08=('fe_flag_w08','sum'),
    fe_retention_w09=('fe_flag_w09','sum'),
    fe_retention_w10=('fe_flag_w10','sum'),
    
    rr_retention_w01=('rr_flag_w01','sum'),
    rr_retention_w02=('rr_flag_w02','sum'),
    rr_retention_w03=('rr_flag_w03','sum'),
    rr_retention_w04=('rr_flag_w04','sum'),
    rr_retention_w05=('rr_flag_w05','sum'),
    rr_retention_w06=('rr_flag_w06','sum'),
    rr_retention_w07=('rr_flag_w07','sum'),
    rr_retention_w08=('rr_flag_w08','sum'),
    rr_retention_w09=('rr_flag_w09','sum'),
    rr_retention_w10=('rr_flag_w10','sum'),
        
    nr_retention_w01=('nr_flag_w01','sum'),
    nr_retention_w02=('nr_flag_w02','sum'),
    nr_retention_w03=('nr_flag_w03','sum'),
    nr_retention_w04=('nr_flag_w04','sum'),
    nr_retention_w05=('nr_flag_w05','sum'),
    nr_retention_w06=('nr_flag_w06','sum'),
    nr_retention_w07=('nr_flag_w07','sum'),
    nr_retention_w08=('nr_flag_w08','sum'),
    nr_retention_w09=('nr_flag_w09','sum'),
    nr_retention_w10=('nr_flag_w10','sum')
    )


# .to_csv(f'{month}/output_files/reg_retention_10w_newlogic.csv',index=False)

#rr_mean_intent_final, rr_regularity_final, reg_intent

# gender_tag,
# taxi_income_segment,
# ps_tag_link,
# ps_tag_auto,
# taxi_service_affinity,
# taxi_distance_preference,
# taxi_time_of_day_preference,
# taxi_day_of_week_preference

,city_name,run_date,reg_intent,user_count,fe_retention_w01,fe_retention_w02,fe_retention_w03,fe_retention_w04,fe_retention_w05,fe_retention_w06,fe_retention_w07,fe_retention_w08,fe_retention_w09,fe_retention_w10,rr_retention_w01,rr_retention_w02,rr_retention_w03,rr_retention_w04,rr_retention_w05,rr_retention_w06,rr_retention_w07,rr_retention_w08,rr_retention_w09,rr_retention_w10,nr_retention_w01,nr_retention_w02,nr_retention_w03,nr_retention_w04,nr_retention_w05,nr_retention_w06,nr_retention_w07,nr_retention_w08,nr_retention_w09,nr_retention_w10
0,Bangalore,2024-04-07,01_daily_01_High,77225,68942,73401,74461,75040,75370,75578,75721,75820,75914,75968,67794,72758,73959,74653,75025,75271,75448,75574,75697,75762,63365,70534,72533,73594,74203,74593,74860,75034,75196,75293
1,Bangalore,2024-04-07,01_daily_02_Medium,33364,28468,31406,32048,32331,32510,32623,32706,32765,32810,32834,27693,30959,31724,32073,32292,32445,32550,32612,32672,32709,24178,29029,30485,31169,31594,31846,32038,32163,32285,32366
2,Bangalore,2024-04-07,02_weekly_01_High,4861,3598,4144,4288,4380,4427,4451,4471,4490,4516,4528,3421,4025,4186,4283,4343,4380,4411,4435,4459,4477,3064,3781,4017,4150,4229,4282,4320,4351,4387,4411
3,Bangalore,2024-04-07,02_weekly_02_Medium,116402,88369,103810,108238,110389,111571,112265,112699,113056,113317,113512,83321,100660,105975,108724,110279,111202,111773,112231,112554,112813,69848,90976,99092,103615,106309,107949,109092,109920,110586,111112
4,Bangalore,2024-04-07,02_weekly_03_Low,35259,21719,27616,29629,30704,31338,31710,31950,32154,32322,32436,19486,25760,28159,29519,30356,30833,31164,31447,31662,31830,15417,22045,25106,27043,28292,29123,29691,30126,30492,30778
5,Bangalore,2024-04-07,03_bi_weekly_01_High,9519,7269,8250,8551,8761,8898,8986,9055,9099,9144,9182,6966,8022,8371,8593,8758,8860,8948,9004,9059,9105,6371,7592,8022,8311,8520,8651,8772,8835,8903,8963
6,Bangalore,2024-04-07,03_bi_weekly_02_Medium,92423,60580,75687,81208,84354,86343,87473,88217,88804,89222,89552,55174,71236,77623,81494,84004,85501,86529,87396,87938,88425,44828,61730,69770,75023,78565,80929,82615,83918,84948,85816
7,Bangalore,2024-04-07,03_bi_weekly_03_Low,91701,50177,68355,76154,80700,83554,85336,86473,87374,88000,88447,43372,61644,70354,75886,79562,81902,83505,84839,85754,86510,32930,50157,59861,66586,71343,74637,77233,79250,80771,82081
8,Bangalore,2024-04-07,04_monthly_01_High,4256,2782,3179,3350,3502,3622,3686,3741,3808,3831,3856,2628,3035,3223,3378,3510,3585,3655,3718,3748,3784,2406,2833,3040,3224,3364,3445,3527,3605,3655,3702
9,Bangalore,2024-04-07,04_monthly_02_Medium,39019,20516,26768,29787,31782,33156,34066,34681,35226,35603,35892,17866,24154,27375,29601,31234,32351,33128,33834,34353,34782,14335,20371,23776,26347,28182,29581,30702,31585,32286,32922


In [20]:
test=list(finaldf)

In [21]:
test=['city']+[i+'_yo' for i in test]

In [22]:
test

['city',
 'city_yo',
 'run_date_yo',
 'reg_intent_yo',
 'base_yo',
 '01_daily_01_High_2024-04-14_yo',
 '01_daily_02_Medium_2024-04-14_yo',
 '02_weekly_01_High_2024-04-14_yo',
 '02_weekly_02_Medium_2024-04-14_yo',
 '02_weekly_03_Low_2024-04-14_yo',
 '03_bi_weekly_01_High_2024-04-14_yo',
 '03_bi_weekly_02_Medium_2024-04-14_yo',
 '03_bi_weekly_03_Low_2024-04-14_yo',
 '04_monthly_01_High_2024-04-14_yo',
 '04_monthly_02_Medium_2024-04-14_yo',
 '04_monthly_03_Low_2024-04-14_yo',
 '05_rare_need_01_High_2024-04-14_yo',
 '05_rare_need_02_Medium_2024-04-14_yo',
 '05_rare_need_03_Low_2024-04-14_yo']

In [23]:
# finaldf.to_csv('stability_v0.csv', index=False)

## Notes

### Explanation of correction_factor Function

The correction_factor function takes two parameters: rr_weight and rapido_age.

Parameters:

rr_weight: A series or array of weights (assumed to be a pandas Series based on the usage).
rapido_age: An integer representing the age in days.

Logic and Calculations:
The function performs different calculations based on the value of rapido_age.

    Condition 1: rapido_age >= 91:
        week_inactive is set to 0.
        multiplier is set to 1.
        This means there is no inactivity, and the multiplier is 1 (no correction).
    
    Condition 2: rapido_age == 0:
        week_inactive is calculated as (91 - rapido_age) // 7 - 1.
        multiplier is calculated using the formula:
            multiplier = rr_weight.sum() / (rr_weight.sum() - (week_inactive * (week_inactive + 1) / 2))
        This corrects for the inactivity based on the number of weeks inactive.
    
    Condition 3: rapido_age % 7 == 0:
        week_inactive is calculated similarly to Condition 2.
        multiplier is calculated using the same formula as in Condition 2.
    
    Condition 4: All Other Cases:
        week_inactive is calculated as (91 - rapido_age) // 7.
        multiplier is calculated using the same formula as in Conditions 2 and 3.
        
    Return Values:
        The function returns a tuple containing:
            multiplier: The correction factor.
            week_inactive: Number of inactive weeks.
            Corrected sum of weights: rr_weight.sum() - (week_inactive * (week_inactive + 1) / 2).
            Corrected sum of squared weights: (rr_weight * rr_weight).sum() - (week_inactive * (week_inactive + 1) * (2 * week_inactive + 1) / 6).

                
Applying correction_factor to fe_data
    The line fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight, x)[0]) adds a new column rr_intent_multiplier to the DataFrame fe_data.

- For each value in the column rapido_age of fe_data, the correction_factor function is called.
- The first element of the returned tuple (i.e., multiplier) is extracted and assigned to the new column rr_intent_multiplier.

In [ ]:
rapido_age = 0
rr_weight = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])

rr_weight_sum = rr_weight.sum()
week_inactive = (91 - rapido_age) // 7 - 1
correction = (week_inactive * (week_inactive + 1)) // 2
adjusted_sum = rr_weight_sum - correction
multiplier = rr_weight_sum / adjusted_sum

print("rr_weight_sum:", rr_weight_sum)
print("week_inactive:", week_inactive)
print("correction:", correction)
print("adjusted_sum:", adjusted_sum)
print("multiplier:", multiplier)

Explanation of Logic

week_inactive: Number of weeks of inactivity calculated based on rapido_age.

correction: Sum of an arithmetic series representing the cumulative impact of inactive weeks.

adjusted_sum: The total sum of rr_weight after adjusting for inactivity.

multiplier: The correction factor used to adjust the weights considering inactivity.

This correction factor ensures that the impact of inactive weeks is accounted for by adjusting the total weight proportionally.

In [ ]:
import numpy as np

rr_weight = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
rr_weight_sum = rr_weight.sum()

rapido_ages = list(range(0, 91, 5))  # List of rapido_age values from 0 to 90 in increments of 5

for rapido_age in rapido_ages:
    if rapido_age >= 91:
        week_inactive = 0
        correction = 0
    else:
        week_inactive = (91 - rapido_age) // 7 - 1
        correction = (week_inactive * (week_inactive + 1)) // 2

    adjusted_sum = rr_weight_sum - correction
    multiplier = rr_weight_sum / adjusted_sum

    print(f"rapido_age: {rapido_age}")
    print("week_inactive:", week_inactive)
    print("correction:", correction)
    print("adjusted_sum:", adjusted_sum)
    print("multiplier:", multiplier)
    print("-" * 40)

Interpretation

rapido_age = 0: The maximum number of inactive weeks (12 weeks) results in a correction of 78, leading to a high multiplier of 7.0.

rapido_age = 5: The number of inactive weeks decreases to 11, with a correction of 66, resulting in a multiplier of approximately 3.64.

rapido_age = 10 to 90: As rapido_age increases, the number of inactive weeks and the corresponding correction decrease, leading to lower multipliers.

For rapido_age values from 80 to 90, the multiplier becomes 1.0, indicating no correction is needed since there are no inactive weeks.